# Import Library

In [ ]:
import pandas as pd
import nltk
import pickle
import os
import string

from nltk import FreqDist
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords, wordnet
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk.tag import pos_tag

from random import shuffle

# Download NLTK Library

In [ ]:
nltk.download("punkt")
nltk.download("wordnet")
nltk.download("stopwords")
nltk.download("averaged_perceptron_tagger")

# Settings Variable

In [ ]:
stemmer = PorterStemmer()
wnl = WordNetLemmatizer()
eng_stopwords = stopwords.words("english")

# Preprocessing

In [ ]:
def preprocessing(document):
    words = word_tokenize(document.lower())
    words = [wnl.lemmatize(word) for word in words]
    words = [stemmer.stem(word) for word in words]

    return {word : True for word in words if word not in eng_stopwords and word.isalpha()}

# Training Model

In [ ]:
def train_model():

    dataset = pd.read_csv("./Dataset/Tweets.csv")

    feature_sets = [(preprocessing(text), label) for text, label in zip(dataset["text"], dataset["airplane_sentiment"])]

    shuffle(feature_sets)

    split_index = int(len(feature_sets)  * 0.85)

    train_set, test_set = feature_sets[:split_index], feature_sets[split_index:]

    classifier = nltk.NaiveBayesClassifier.train(train_set)

    accuracy = nltk.classify.accuracy(classifier, test_set)

    print (f"Accuracy : {accuracy}")

    classifier.show_most_informative_features(5)

    file = open ("model.pickle", "wb")
    pickle.dump(file)
    file.close()

    return classifier

def read_model():

    if os.path.exists("./model.pickle"):
        file = open("model.pickle", "rb")
        classifier = pickle.load(file)
        file.close()
        classifier.show_most_informative_features(5)
        print ("Model load successfully")
    
    else:
        print ("Model not found. Training the model...")
        classifier = train_model()
    
    return classifier

# Functions

In [ ]:
def write_review():
    while True:
        review = input("Please input your review")
        words = review.split()

        if len (words) > 1:
            print ("Review added successfully")
            return review
        else:
            print ("Please input your review with more than 1 word")

def analyze_review(review, classifier):

    if len(review) == 0:
        print ("Please input your review first")
        return

    words = word_tokenize(review.lower())

    words = FreqDist([word for word in words if word.isalpha() and word not in string.punctuation])

    tagged = pos_tag(words)

    # POS Tagging

    print ("Review Part of POS Tagging")
    for i, word in enumerate(tagged):
        print (f"{i+1}. {word[0]}, {word[1]}")
    
    # Synonyms and Antonyms

    for word in words:

        print ("=========================")
        print (f"Word : {word}")
        print ("=========================")

        synsets = wordnet.synsets(word)

        synonyms = []
        antonyms = []

        for synset in synsets:
            for lemma in synset.lemmas():
                synonyms.append(lemma.name())
                for antonym in lemma.antonyms():
                    antonyms.append(antonym.name())

        print ("Synonyms")
        if len(synonyms) == 0:
            print ("No synonyms")
        else:
            for syn in synonyms[:5]:
                print (f"(+) {syn}")
        
        print ("Antonyms")
        if len(antonyms) == 0:
            print ("No antonyms")
        else:
            for ant in antonyms[:5]:
                print (f"(-) {ant}")

        print("===========================")

    # Category

    clean_review = [word for word in word_tokenize(review) if word not in string.punctuation and word not in eng_stopwords]

    clean_review = [wnl.lemmatize(stemmer.stem(word)) for word in clean_review]

    result = classifier.classify(FreqDist(clean_review))

    print (f"Your Review : {review}")
    print (f"Your Review Category : {result}")


# Main Functions

In [ ]:
def mainMenu():
    
	classifier = read_model()

	review = ""

	while True:
		print ("Tweet Sentiment Analysis")
		print ("Your review: ", "No Review" if len(review) == 0 else review)
		print ("1. Input Review")
		print ("2. Analyze Your Review")
		print ("3. Exit")
		print (">> ")

		choice = input("Please input your menu choice")
		if choice == '1':
			review = write_review()
		elif choice == '2':
			analyze_review(review, classifier)
		elif choice == '3':
			print ("Thank you :)")
			break
		else:
			print ("Input Invalid! Please choose menu choice between 1-3")

In [ ]:
mainMenu()